# IOS Risk: three-capability candidate

One fixed candidate; 1,900 training rows, 238 optimizer steps, 490 evaluations per model. Outputs require offline semantic review. No automatic release, retry or publication.

In [ ]:
try:
    from pathlib import Path
    import hashlib, json, os, subprocess, sys, time, uuid

    SESSION_START = time.monotonic()
    SESSION_CEILING = 21600
    MANIFEST_SHA256 = "fd9bbed76768cc2e6bcdc85527d2ab971813af84ea4d7e7e6f80f6e1164de1c6"
    EXPECTED = {
        "runtime.zip": "8a288794a2de97817a3a2a5f60935008f656699faab6c6c32b60a5974586d1a0",
        "candidate_data.json": "5e3f89628937d4b033eddc08f3f9fbf0422a53c30b3e2675b90d644dd60373de",
        "token_checks.json": "7ed0bccd4061d6e5e841735761f964c42df9443b870f850219d381176485b891",
        "review-regulatory-blind-packet-v1.json": "1d95040009d945463779909513d158fc835c0b70f5a33fcf15502b7a82d7db37",
        "review-regulatory-blind-packet-v2.json": "99ad0fd626ebca78b79945ebcdd6d0874124881a697c8c71a553af7e02363270",
        "review-regulatory-blind-proposals-v1.json": "1c57b40d9f34c1e5fccc37d6dfede534cb65c9921243e0a69775e1e6d122b0e5",
        "review-regulatory-blind-proposals-v2.json": "0ab09de8bbd27f1b0eac36dd662e58ec8c95cebdbf9ad026c3105bb9deb72ce2",
        "review-risk-blind-packet-v1.json": "dbcac9a1e55985c8177a37e528581a583db924bdcdef8138b25a770473e7ac0e",
        "review-risk-blind-packet-v2.json": "2d62dacb6fd2faea24fadedaa232719e8f0d79f80a21bb58ccf388c10bc72a03",
        "review-risk-blind-proposals-v1.json": "7d4efdb0bc6e5c3eedba065b32f307d4bc3ee3d8174cc41fb6faed925a25590d",
        "review-risk-blind-proposals-v2.json": "2117a24fec67417f2cba0b5acd255dc7020b9c253bc851f1a7a1a5aaecb90a84",
        "classification_source_receipt.json": "6de1cd82440a2f7d22f8f2174bac5d9f68563ccdbc42bd29e3fe1db256876518",
    }
    roots = [
        p.parent
        for p in Path("/kaggle/input").rglob("manifest.json")
        if hashlib.sha256(p.read_bytes()).hexdigest() == MANIFEST_SHA256
    ]
    if len(roots) != 1:
        raise ValueError("Attach exactly one matching private candidate dataset")
    if list(Path("/kaggle/input").rglob("adapter_model.safetensors")) or list(
        Path("/kaggle/input").rglob("adapter_config.json")
    ):
        raise ValueError("Old adapter mounts are forbidden")
    REMOTE_BUNDLE = roots[0]
    manifest = json.loads((REMOTE_BUNDLE / 'manifest.json').read_text())
    remote_expected = (set(EXPECTED) - {'runtime.zip'}) | {'manifest.json'} | {'runtime/' + name for name in manifest['source_files']}
    remote_actual = {str(p.relative_to(REMOTE_BUNDLE)) for p in REMOTE_BUNDLE.rglob('*') if p.is_file()}
    if remote_actual - remote_expected - {'dataset-metadata.json'} or remote_expected - remote_actual:
        raise ValueError('Unexpected expanded dataset layout')
    for name, expected in manifest['source_files'].items():
        if hashlib.sha256((REMOTE_BUNDLE / 'runtime' / name).read_bytes()).hexdigest() != expected:
            raise ValueError('Expanded runtime source changed: ' + name)
    import base64
    BUNDLE = Path('/kaggle/working') / ('verified-bundle-' + uuid.uuid4().hex[:12])
    BUNDLE.mkdir(exist_ok=False)
    for name in set(EXPECTED) - {'runtime.zip'} | {'manifest.json'}:
        raw = (REMOTE_BUNDLE / name).read_bytes()
        expected = MANIFEST_SHA256 if name == 'manifest.json' else EXPECTED[name]
        if hashlib.sha256(raw).hexdigest() != expected:
            raise ValueError('Remote data changed: ' + name)
        with (BUNDLE / name).open('xb') as staged:
            staged.write(raw)
    archive_bytes = base64.b64decode('...', validate=True)
    if hashlib.sha256(archive_bytes).hexdigest() != EXPECTED['runtime.zip']:
        raise ValueError('Embedded approved archive changed')
    with (BUNDLE / 'runtime.zip').open('xb') as staged:
        staged.write(archive_bytes)
    if (
        {str(p.relative_to(BUNDLE)) for p in BUNDLE.rglob("*") if p.is_file()}
        - set(EXPECTED)
        - {"manifest.json", "dataset-metadata.json"}
    ):
        raise ValueError("Unexpected files in candidate attachment")
    for name, expected in EXPECTED.items():
        if hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest() != expected:
            raise ValueError("Changed package bytes: " + name)
    OUTPUT = Path("/kaggle/working") / ("candidate-" + uuid.uuid4().hex[:12])
    COMMAND = [
        sys.executable,
        "-I",
        "-B",
        "-c",
        "import sys,runpy; sys.path.insert(0,sys.argv.pop(1)+'/runtime.zip'); runpy.run_module('train.release_candidate.runner',run_name='__main__')",
        str(BUNDLE),
        "--bundle",
        str(BUNDLE),
        "--manifest-sha256",
        MANIFEST_SHA256,
    ]
    subprocess.run(COMMAND, check=True, timeout=120)
    print("Exact package and CPU data checks passed; no model loaded")

except BaseException as cell_error:
    import json, uuid
    from pathlib import Path

    with (
        Path("/kaggle/working") / ("cell-failure-" + uuid.uuid4().hex[:12] + ".json")
    ).open("x") as failure_file:
        json.dump(
            {
                "status": "FAILED_NO_RETRY",
                "error": type(cell_error).__name__,
                "detail": str(cell_error),
            },
            failure_file,
        )
    raise

In [ ]:
try:
    import subprocess
    import sys

    DEPENDENCY_PINS = {
        "transformers": "5.5.0",
        "datasets": "4.3.0",
        "trl": "0.24.0",
        "bitsandbytes": "0.50.1",
        "xformers": "0.0.34",
        "peft": "0.19.1",
        "unsloth": "2026.8.22",
        "unsloth_zoo": "2026.8.16",
    }

    # No shell pipeline hiding pip failures. No private repo clone or interactive login.
    requirements = [f"{name}=={version}" for name, version in DEPENDENCY_PINS.items()]
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--no-input",
            "--retries",
            "0",
            "--timeout",
            "60",
            "torch==2.10.0",
            *requirements,
        ],
        check=True,
        timeout=600,
    )
    # Verify in a fresh interpreter: the notebook kernel may hold pre-install imports.
    probe = """
    import importlib.metadata as md
    import json
    import sys
    expected = json.loads(sys.argv[1])
    actual = {name: md.version(name) for name in expected}
    if actual != expected:
        raise RuntimeError(f"Dependency mismatch: {actual}")
    import torch
    if torch.__version__.split("+")[0] != "2.10.0":
        raise RuntimeError(f"Unexpected torch version: {torch.__version__}")
    if not torch.cuda.is_available() or torch.cuda.get_device_capability(0)[0] < 7:
        raise RuntimeError("Select GPU T4 x2; CUDA capability 7+ is required")
    print("[Preflight] pins and GPU verified:", actual, torch.cuda.get_device_name(0))
    """
    subprocess.run(
        [sys.executable, "-I", "-c", __import__("textwrap").dedent(probe), json.dumps(DEPENDENCY_PINS)],
        check=True,
        timeout=60,
    )

except BaseException as cell_error:
    import json, uuid
    from pathlib import Path

    with (
        Path("/kaggle/working") / ("cell-failure-" + uuid.uuid4().hex[:12] + ".json")
    ).open("x") as failure_file:
        json.dump(
            {
                "status": "FAILED_NO_RETRY",
                "error": type(cell_error).__name__,
                "detail": str(cell_error),
            },
            failure_file,
        )
    raise

In [ ]:
try:
    remaining = min(21000, SESSION_CEILING - (time.monotonic() - SESSION_START) - 30)
    if remaining <= 0:
        raise RuntimeError("Session budget exhausted before training")
    try:
        subprocess.run(
            COMMAND + ["--output", str(OUTPUT), "--run"], check=True, timeout=remaining
        )
    except BaseException as exc:
        failure = Path("/kaggle/working") / (
            "supervisor-failure-" + uuid.uuid4().hex[:12] + ".json"
        )
        failure.write_text(
            json.dumps(
                {
                    "status": "FAILED_NO_RETRY",
                    "error": type(exc).__name__,
                    "detail": str(exc),
                    "output": str(OUTPUT),
                    "elapsed_seconds": time.monotonic() - SESSION_START,
                }
            )
        )
        raise
    if time.monotonic() - SESSION_START > SESSION_CEILING:
        raise RuntimeError("Session ceiling exceeded")
    with (OUTPUT / "supervisor_receipt.json").open("x") as handle:
        json.dump(
            {
                "manifest_sha256": MANIFEST_SHA256,
                "session_seconds": time.monotonic() - SESSION_START,
                "session_ceiling": SESSION_CEILING,
                "worker_timeout_seconds": remaining,
                "completed": True,
            },
            handle,
        )
    print((OUTPUT / "result_receipt.json").read_text())

except BaseException as cell_error:
    import json, uuid
    from pathlib import Path

    with (
        Path("/kaggle/working") / ("cell-failure-" + uuid.uuid4().hex[:12] + ".json")
    ).open("x") as failure_file:
        json.dump(
            {
                "status": "FAILED_NO_RETRY",
                "error": type(cell_error).__name__,
                "detail": str(cell_error),
            },
            failure_file,
        )
    raise